# 局域基元生成：并行阈值优化版（LCMWR/scripts 路径适配）

本 notebook 在“并行阈值优化版”基础上继续调整路径规则：

1. 默认假定脚本/notebook 放在 `LCMWR/scripts` 下。
2. 输入默认读取 `../data/unique_smiles_for_fragments.csv`，即 `LCMWR/data/unique_smiles_for_fragments.csv`。
3. 输出默认写入 `../results/`，即 `LCMWR/results/`。
4. 同时兼容从 `motif/`、`LCMWR/scripts/` 或项目根目录启动 notebook 的情况。
5. `[Fr]` 和 `[Rb]` 仍按原逻辑替换为 `[H]`。
6. 不使用 batch；候选以 canonical fragment SMILES 为唯一键，已生成结构不会重复添加，只合并来源标签。
7. 候选生成与全局频率统计分离。
8. 能并行时使用 `n_jobs=-1`，候选生成和频率统计都支持 joblib 并行；若 joblib 不可用则自动回退到串行。
9. 最终保留机制：只包含 C/H/O/N 的片段 `support_ratio >= 1%`；包含其他元素的片段 `support_ratio >= 0.5%`。
10. 保留芳香环/脂肪环补全机制，避免半截环片段。
11. 默认关闭全量可视化；如需可视化，只保存 Top-N 片段图。


In [10]:
# =========================
# 1. 参数配置
# =========================
from dataclasses import dataclass, asdict, field
import hashlib
from pathlib import Path
from typing import Optional, Tuple

GENERATOR_VERSION = 'ring_complete_frequency_cache_v3'

@dataclass
class FragmentConfig:
    # 路径说明：本脚本/notebook 默认位于 LCMWR/scripts 下。
    # 因此输入 ../data/... 对应 LCMWR/data/...，输出 ../results/... 对应 LCMWR/results/...
    # 如果从其他工作目录启动，后续 resolve_motif_path() 会自动尝试定位 motif 根目录。
    motif_root: Optional[Path] = None          # 通常不需要填写；必要时可手动指定为 .../motif
    input_file: Path = Path("../data/unique_smiles_for_fragments.csv")
    smiles_column: Optional[str] = None       # None 时自动识别 SMILES 列
    output_file: Path = Path("../results/local_vocab_parallel_threshold.csv")
    stats_file: Path = Path("../results/local_vocab_parallel_threshold_stats.json")
    visualization_dir: Path = Path("../results/fragment_viz")

    # 全量频率统计缓存：保存所有候选的 support/occurrence，改筛选阈值时可直接重筛。
    use_frequency_cache: bool = True
    frequency_cache_file: Path = Path("../results/local_vocab_full_frequency_cache.csv")
    frequency_cache_stats_file: Path = Path("../results/local_vocab_full_frequency_cache_stats.json")

    # 默认重新计算，避免旧 CSV 绕过环补全与支持度规则。
    use_result_cache: bool = True
    result_cache_files: Tuple[Path, ...] = (
        Path("../results/local_vocab_parallel_threshold.csv"),
    )

    # 片段尺寸参数
    min_size: int = 1
    max_size: int = 17                    # 环补全后仍需满足该原子数上限

    # 支持度保留机制
    chon_elements: Tuple[str, ...] = ("C", "H", "O", "N")
    chon_min_support_ratio: float = 0.01      # 只含 CHON 的片段至少覆盖 1% 分子
    other_min_support_ratio: float = 0.005    # 含其他元素的片段至少覆盖 0.5% 分子
    min_support_floor: int = 1                # 小数据集时至少保留出现过 1 个分子的片段

    # 候选生成控制参数
    # 默认不设置全局候选数上限，避免“先生成的候选”截断后续分子。
    max_candidates: Optional[int] = None      # None 表示不限制；如设置，只在全局合并阶段截断
    max_candidates_per_mol: int = 3000        # 单个分子最多贡献多少个不同候选片段
    max_atom_sets_per_mol: int = 8000        # 单个分子最多探索多少个原子集合

    # 并行参数
    n_jobs: int = -1                          # -1 表示使用所有可用 CPU
    parallel_prefer: str = "threads"           # 可选："threads" 或 "processes"；默认 threads 避免 notebook 环境下多进程导入问题

    # 环结构处理
    include_rings: bool = True                # 显式加入完整环候选，包括芳香环和脂肪环
    include_aromatic_rings: bool = True       # 额外给完整芳香环打 aromatic_ring 来源标签
    complete_partial_rings: bool = True       # 连通扩展命中部分环时，生成候选前自动补全完整环
    complete_aromatic_rings: bool = True      # 补全芳香环
    complete_aliphatic_rings: bool = True     # 补全脂肪环/非芳香环

    # 行为开关
    replace_ports_with_h: bool = True         # 按原逻辑将 [Fr]/[Rb] 替换为 [H]
    visualize: bool = False                  # 默认关闭可视化
    top_n_viz: int = 50                      # visualize=True 时只画 Top-N

    # 日志
    log_level: str = "INFO"

CONFIG = FragmentConfig()
CONFIG


FragmentConfig(motif_root=None, input_file=PosixPath('../data/unique_smiles_for_fragments.csv'), smiles_column=None, output_file=PosixPath('../results/local_vocab_parallel_threshold.csv'), stats_file=PosixPath('../results/local_vocab_parallel_threshold_stats.json'), visualization_dir=PosixPath('../results/fragment_viz'), use_frequency_cache=True, frequency_cache_file=PosixPath('../results/local_vocab_full_frequency_cache.csv'), frequency_cache_stats_file=PosixPath('../results/local_vocab_full_frequency_cache_stats.json'), use_result_cache=True, result_cache_files=(PosixPath('../results/local_vocab_parallel_threshold.csv'),), min_size=1, max_size=17, chon_elements=('C', 'H', 'O', 'N'), chon_min_support_ratio=0.01, other_min_support_ratio=0.005, min_support_floor=1, max_candidates=None, max_candidates_per_mol=3000, max_atom_sets_per_mol=8000, n_jobs=-1, parallel_prefer='threads', include_rings=True, include_aromatic_rings=True, complete_partial_rings=True, complete_aromatic_rings=True, c

In [11]:
# =========================
# 2. 导入依赖与日志设置
# =========================
import gc
import json
import logging
import math
from collections import defaultdict, deque
from typing import Dict, Iterable, List, Optional, Set, Tuple

import pandas as pd
from rdkit import Chem, RDLogger, rdBase
from rdkit.Chem import Draw

try:
    from joblib import Parallel, delayed
    JOBLIB_AVAILABLE = True
except Exception:
    Parallel = None
    delayed = None
    JOBLIB_AVAILABLE = False

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

# 屏蔽 RDKit C++ 层日志，避免 kekulize/parse error 在 notebook 中刷屏。
for _rdkit_log_name in ("rdApp.debug", "rdApp.info", "rdApp.warning", "rdApp.error", "rdApp.*"):
    RDLogger.DisableLog(_rdkit_log_name)
rdBase.DisableLog("rdApp.*")
RDLogger.logger().setLevel(RDLogger.CRITICAL)

logger = logging.getLogger("local_fragment_mining")

def setup_logging(level: str = "INFO") -> None:
    """设置 notebook 友好的 logging 输出。"""
    logger.handlers.clear()
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
    logger.addHandler(handler)
    logger.setLevel(getattr(logging, level.upper(), logging.INFO))

setup_logging(CONFIG.log_level)
logger.info("joblib 可用: %s", JOBLIB_AVAILABLE)

[INFO] joblib 可用: True


In [12]:
# =========================
# 3. SMILES 读取与预处理
# =========================
COMMON_SMILES_COLUMNS = [
    "smiles", "SMILES", "canonical_smiles", "canonical_p_smiles",
    "p_smiles", "P_SMILES", "unique_smiles", "polymer_smiles"
]


def infer_motif_root(config: FragmentConfig) -> Path:
    """
    自动推断 motif 根目录。

    兼容三种常见启动位置：
    1. 在 LCMWR/scripts 下启动；
    2. 在 motif 下启动；
    3. 在 motif 的上一级项目目录下启动。
    """
    if config.motif_root is not None:
        return Path(config.motif_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "LCMWR", *cwd.parents]
    for root in candidates:
        if root.name.lower() == "lcmwr" and all((root / part).exists() for part in ("data", "results", "scripts")):
            return root
    raise FileNotFoundError(f"无法定位 LCMWR 根目录：{cwd}")


def resolve_motif_path(path_like, config: FragmentConfig, must_exist: bool = False) -> Path:
    """
    解析输入/输出路径。

    相对路径优先按“脚本位于 LCMWR/scripts”解释：
    - ../data/xxx  → LCMWR/data/xxx
    - ../results/xxx → LCMWR/results/xxx

    同时兼容旧写法 LCMWR/data/xxx 和从不同工作目录启动的情况。
    """
    path = Path(path_like).expanduser()
    if path.is_absolute():
        return path

    motif_root = infer_motif_root(config)
    scripts_dir = motif_root / "scripts"
    cwd = Path.cwd().resolve()

    candidates = [
        (scripts_dir / path).resolve(),  # 默认：相对 LCMWR/scripts
        (motif_root / path).resolve(),    # 兼容 data/xxx、results/xxx
        (cwd / path).resolve(),          # 兼容当前工作目录相对路径
    ]

    # 兼容旧配置：LCMWR/data/xxx 或 LCMWR/results/xxx
    if path.parts and path.parts[0] == "LCMWR":
        candidates.insert(0, (motif_root.parent / path).resolve())

    if must_exist:
        for candidate in candidates:
            if candidate.exists():
                return candidate

    # 输出文件通常还不存在，因此默认返回相对 LCMWR/scripts 解释后的路径。
    return candidates[0]


def get_resolved_paths(config: FragmentConfig) -> Dict[str, object]:
    """返回实际解析后的关键路径，便于日志与 stats 记录。"""
    cache_files = [
        str(resolve_motif_path(path, config, must_exist=True))
        for path in getattr(config, "result_cache_files", tuple())
    ]
    return {
        "motif_root": str(infer_motif_root(config)),
        "input_file": str(resolve_motif_path(config.input_file, config, must_exist=True)),
        "frequency_cache_file": str(resolve_motif_path(config.frequency_cache_file, config)),
        "frequency_cache_stats_file": str(resolve_motif_path(config.frequency_cache_stats_file, config)),
        "result_cache_files": cache_files,
        "output_file": str(resolve_motif_path(config.output_file, config)),
        "stats_file": str(resolve_motif_path(config.stats_file, config)),
        "visualization_dir": str(resolve_motif_path(config.visualization_dir, config)),
    }

def preprocess_smiles(smiles: str, replace_ports_with_h: bool = True) -> str:
    """按原逻辑处理聚合物端口标记：将 [Fr]/[Rb] 替换为 [H]。"""
    if not isinstance(smiles, str):
        return ""
    smiles = smiles.strip()
    if replace_ports_with_h:
        smiles = smiles.replace("[Fr]", "[H]").replace("[Rb]", "[H]")
    return smiles


def infer_smiles_column(df: pd.DataFrame, smiles_column: Optional[str] = None) -> str:
    """自动识别输入表中的 SMILES 列。"""
    if smiles_column is not None:
        if smiles_column not in df.columns:
            raise ValueError(f"指定的 smiles_column={smiles_column!r} 不在输入文件列名中。")
        return smiles_column

    for col in COMMON_SMILES_COLUMNS:
        if col in df.columns:
            return col

    object_cols = [col for col in df.columns if df[col].dtype == "object"]
    if len(object_cols) == 1:
        return object_cols[0]

    raise ValueError(
        "无法自动识别 SMILES 列。请在 CONFIG.smiles_column 中指定列名。"
        f"当前列名为: {list(df.columns)}"
    )


def load_molecule_records(config: FragmentConfig) -> List[Dict]:
    """读取输入文件，返回 original_smiles、processed_smiles、mol 的记录列表。"""
    input_file = resolve_motif_path(config.input_file, config, must_exist=True)
    if not input_file.exists():
        raise FileNotFoundError(f"输入文件不存在: {input_file}")

    df = pd.read_csv(input_file)
    smiles_col = infer_smiles_column(df, config.smiles_column)
    logger.info("读取输入文件: %s", input_file)
    logger.info("使用 SMILES 列: %s", smiles_col)

    records = []
    invalid = 0
    empty = 0
    duplicated_after_processing = 0
    seen_processed = set()

    for raw in df[smiles_col].tolist():
        processed = preprocess_smiles(raw, config.replace_ports_with_h)
        if not processed:
            empty += 1
            continue

        # 输入文件本身是 unique smiles；这里再对处理后的 SMILES 去重，避免 [Fr]/[Rb] 替换后重复。
        if processed in seen_processed:
            duplicated_after_processing += 1
            continue
        seen_processed.add(processed)

        mol = Chem.MolFromSmiles(processed)
        if mol is None:
            invalid += 1
            continue

        records.append({
            "original_smiles": raw,
            "processed_smiles": processed,
            "mol": mol,
        })

    logger.info(
        "原始行数: %d；有效分子数: %d；空值: %d；无效 SMILES: %d；处理后重复: %d",
        len(df), len(records), empty, invalid, duplicated_after_processing,
    )
    return records

In [13]:
# =========================
# 4. 片段提取工具函数
# =========================
def cleanup_memory() -> None:
    gc.collect()


def atom_rings(mol: Chem.Mol) -> List[Tuple[int, ...]]:
    """获取分子中的 SSSR 环。"""
    if mol is None:
        return []
    return [tuple(ring) for ring in mol.GetRingInfo().AtomRings()]


def is_aromatic_atom_set(mol: Chem.Mol, atom_indices: Iterable[int]) -> bool:
    """判断原子集合是否全由芳香原子构成。"""
    return all(mol.GetAtomWithIdx(int(idx)).GetIsAromatic() for idx in atom_indices)


def ring_source_label(mol: Chem.Mol, ring: Iterable[int]) -> str:
    """返回环类型来源标签：aromatic_ring 或 aliphatic_ring。"""
    return "aromatic_ring" if is_aromatic_atom_set(mol, ring) else "aliphatic_ring"


def complete_partial_rings(
    mol: Chem.Mol,
    atom_indices: Iterable[int],
    config: FragmentConfig,
) -> Tuple[List[int], Set[str]]:
    """
    如果候选原子集合只包含某个环的一部分，则补全该完整环。

    说明：
    - 只根据原始 atom_indices 判断是否命中部分环，避免补全一个环后无限级联扩展到整个稠环体系。
    - 显式 ring 候选调用 add_candidate(..., complete_rings=False)，不会因为稠环共享原子而被动扩成更大的稠环片段。
    - 环补全后如果超过 config.max_size，会在 add_candidate 中被丢弃。
    """
    atom_set = {int(i) for i in atom_indices}
    original_set = set(atom_set)
    completion_sources: Set[str] = set()

    if not getattr(config, "complete_partial_rings", True):
        return sorted(atom_set), completion_sources

    for ring in atom_rings(mol):
        ring_set = set(ring)
        overlap = original_set & ring_set
        if not overlap or len(overlap) == len(ring_set):
            continue

        label = ring_source_label(mol, ring)
        if label == "aromatic_ring" and not getattr(config, "complete_aromatic_rings", True):
            continue
        if label == "aliphatic_ring" and not getattr(config, "complete_aliphatic_rings", True):
            continue

        atom_set.update(ring_set)
        completion_sources.add(f"completed_{label}")
        completion_sources.add("ring_completed")

    return sorted(atom_set), completion_sources


def normalize_fragment_smiles(smiles: str) -> Optional[str]:
    """将片段 SMILES 规范化为可解析、canonical、Kekule 风格的 SMILES。"""
    if not smiles:
        return None
    try:
        frag_mol = Chem.MolFromSmiles(smiles)
        if frag_mol is None:
            return None
        return Chem.MolToSmiles(
            frag_mol,
            canonical=True,
            isomericSmiles=True,
            kekuleSmiles=True,
        )
    except Exception:
        return None


def fragment_smiles_from_atoms(mol: Chem.Mol, atom_indices: Iterable[int]) -> Optional[str]:
    """
    从原分子的一组原子索引提取连通片段 SMILES。

    关键优化：使用 kekuleSmiles=True，避免从芳香环中截取局部片段时生成
    RDKit 难以规范化的 c/cc/ccc 等非环芳香原子片段。
    """
    atom_indices = sorted({int(i) for i in atom_indices})
    if not atom_indices:
        return None
    try:
        smiles = Chem.MolFragmentToSmiles(
            mol,
            atomsToUse=atom_indices,
            canonical=True,
            isomericSmiles=True,
            kekuleSmiles=True,
        )
        return normalize_fragment_smiles(smiles)
    except Exception:
        return None


def fragment_mol_from_smiles(smiles: str) -> Optional[Chem.Mol]:
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None


def fragment_elements(fragment_mol: Chem.Mol) -> List[str]:
    """返回片段中显式原子的元素集合。RDKit 隐式 H 不作为独立原子统计。"""
    if fragment_mol is None:
        return []
    return sorted({atom.GetSymbol() for atom in fragment_mol.GetAtoms()})


def is_chon_only_fragment(fragment_mol: Chem.Mol, config: FragmentConfig) -> bool:
    """判断片段显式原子是否只包含 C/H/O/N。"""
    allowed = set(config.chon_elements)
    elements = fragment_elements(fragment_mol)
    return bool(elements) and all(element in allowed for element in elements)


def support_threshold_for_fragment(fragment_mol: Chem.Mol, n_mols: int, config: FragmentConfig) -> Tuple[str, float, int]:
    """根据元素组成返回动态支持度阈值。"""
    if is_chon_only_fragment(fragment_mol, config):
        threshold_type = "CHON_only"
        ratio = config.chon_min_support_ratio
    else:
        threshold_type = "contains_other_elements"
        ratio = config.other_min_support_ratio

    min_count = max(int(config.min_support_floor), int(math.ceil(n_mols * ratio)))
    return threshold_type, ratio, min_count


def fragment_ring_flags(fragment_mol: Chem.Mol, smiles: str = "") -> Tuple[bool, bool, bool]:
    """返回 has_ring, has_aromatic_ring, has_aliphatic_ring。"""
    if fragment_mol is None:
        return False, False, False

    rings = fragment_mol.GetRingInfo().AtomRings()
    has_ring = len(rings) > 0
    has_aromatic_ring = False
    has_aliphatic_ring = False

    for ring in rings:
        if is_aromatic_atom_set(fragment_mol, ring):
            has_aromatic_ring = True
        else:
            has_aliphatic_ring = True

    # 兜底：如果 query 中存在芳香键符号，也视为含芳香环/芳香结构。
    if ":" in smiles:
        has_aromatic_ring = True

    return has_ring, has_aromatic_ring, has_aliphatic_ring


def fragment_type_flags(fragment_mol: Chem.Mol, smiles: str) -> Tuple[bool, bool]:
    """兼容旧输出字段：返回 is_ring, is_aromatic。"""
    has_ring, has_aromatic_ring, _ = fragment_ring_flags(fragment_mol, smiles)
    return has_ring, has_aromatic_ring

In [14]:
# =========================
# 5. 候选片段生成：不使用 batch，已生成结构不重复添加
# =========================
def add_candidate(
    candidate_sources: Dict[str, Set[str]],
    mol: Chem.Mol,
    atom_indices: Iterable[int],
    source,
    config: FragmentConfig,
    complete_rings: bool = True,
) -> Tuple[Optional[str], bool]:
    """
    从原子集合生成候选片段并记录来源。

    返回：
    - smiles：候选片段 SMILES；失败时为 None。
    - is_new：该结构是否第一次加入当前候选字典。

    说明：候选以 canonical fragment SMILES 为唯一键；如果结构已经生成过，不再重复添加，
    只合并 source_types，保证最终每个片段只有一条候选记录。
    """
    atom_indices = sorted({int(i) for i in atom_indices})

    if isinstance(source, str):
        source_set = {source}
    else:
        source_set = {str(s) for s in source if s}

    if complete_rings:
        atom_indices, completion_sources = complete_partial_rings(mol, atom_indices, config)
        source_set.update(completion_sources)

    if not (config.min_size <= len(atom_indices) <= config.max_size):
        return None, False

    smiles = fragment_smiles_from_atoms(mol, atom_indices)
    if smiles is None:
        return None, False

    frag_mol = fragment_mol_from_smiles(smiles)
    if frag_mol is None:
        return None, False

    n_atoms = frag_mol.GetNumAtoms()
    if not (config.min_size <= n_atoms <= config.max_size):
        return None, False

    is_new = smiles not in candidate_sources
    candidate_sources[smiles].update(sorted(source_set))
    return smiles, is_new


def generate_candidates_for_smiles(
    processed_smiles: str,
    config: FragmentConfig,
) -> Dict:
    """对单个 SMILES 生成启发式局域候选片段；该函数可被 joblib 并行调用。"""
    mol = Chem.MolFromSmiles(processed_smiles)
    if mol is None:
        return {
            "processed_smiles": processed_smiles,
            "candidate_sources": {},
            "local_candidate_count": 0,
            "explored_atom_set_count": 0,
            "ring_completed_candidate_count": 0,
            "skipped_existing_candidate_count": 0,
            "failed": True,
        }

    candidate_sources: Dict[str, Set[str]] = defaultdict(set)
    explored_atom_sets = 0
    ring_completed_candidate_count = 0
    skipped_existing_candidate_count = 0

    # 1) 显式加入完整环候选，避免普通 BFS 截断时漏掉完整环。
    #    显式环候选不做环补全，避免稠环中的一个 SSSR 环被自动扩成更大的稠环片段。
    if config.include_rings or config.include_aromatic_rings:
        for ring in atom_rings(mol):
            label = ring_source_label(mol, ring)
            sources = []
            if config.include_rings:
                sources.extend(["ring", label])
            elif config.include_aromatic_rings and label == "aromatic_ring":
                sources.append("aromatic_ring")

            if sources:
                smiles, is_new = add_candidate(candidate_sources, mol, ring, sources, config, complete_rings=False)
                if smiles and not is_new:
                    skipped_existing_candidate_count += 1

    # 2) 连通子图扩展：从每个原子作为 seed，逐步加入邻接原子。
    #    如果扩展出的原子集合命中了部分芳香环/脂肪环，add_candidate 会先补全该环再生成候选。
    queue = deque(frozenset([atom.GetIdx()]) for atom in mol.GetAtoms())
    seen_atom_sets: Set[frozenset] = set()

    while queue:
        atom_set = queue.popleft()
        if atom_set in seen_atom_sets:
            continue
        seen_atom_sets.add(atom_set)
        explored_atom_sets += 1

        if explored_atom_sets > config.max_atom_sets_per_mol:
            break
        if len(candidate_sources) >= config.max_candidates_per_mol:
            break

        source = "atom_seed" if len(atom_set) == 1 else "connected_extension"
        smiles, is_new = add_candidate(candidate_sources, mol, atom_set, source, config, complete_rings=True)
        if smiles:
            if not is_new:
                skipped_existing_candidate_count += 1
            if "ring_completed" in candidate_sources[smiles]:
                ring_completed_candidate_count += 1

        if len(atom_set) >= config.max_size:
            continue

        neighbor_indices = set()
        for idx in atom_set:
            atom = mol.GetAtomWithIdx(int(idx))
            for neighbor in atom.GetNeighbors():
                nb_idx = neighbor.GetIdx()
                if nb_idx not in atom_set:
                    neighbor_indices.add(nb_idx)

        for nb_idx in sorted(neighbor_indices):
            next_set = frozenset(set(atom_set) | {nb_idx})
            if next_set not in seen_atom_sets:
                queue.append(next_set)

    return {
        "processed_smiles": processed_smiles,
        "candidate_sources": {smiles: sorted(sources) for smiles, sources in candidate_sources.items()},
        "local_candidate_count": len(candidate_sources),
        "explored_atom_set_count": explored_atom_sets,
        "ring_completed_candidate_count": ring_completed_candidate_count,
        "skipped_existing_candidate_count": skipped_existing_candidate_count,
        "failed": False,
    }


def _parallel_or_sequential(items, func, config: FragmentConfig, desc: str) -> List:
    """joblib 可用且 n_jobs != 1 时并行；否则串行。"""
    if JOBLIB_AVAILABLE and config.n_jobs != 1:
        logger.info("%s：使用 joblib 并行，n_jobs=%s，prefer=%s", desc, config.n_jobs, config.parallel_prefer)
        return Parallel(n_jobs=config.n_jobs, prefer=config.parallel_prefer)(
            delayed(func)(item, config) for item in items
        )

    logger.info("%s：使用串行模式", desc)
    return [func(item, config) for item in items]


def generate_candidate_fragments(
    records: List[Dict],
    config: FragmentConfig,
) -> Tuple[Dict[str, Set[str]], Dict]:
    """
    全局候选生成。

    变化点：
    - 不使用 batch。
    - 每个分子独立生成候选，随后全局合并。
    - 全局候选以 fragment SMILES 为唯一键；已生成结构不重复添加，只合并来源标签。
    - 不按支持度过滤，支持度只在后续全局统计后统一应用。
    """
    processed_smiles_list = [rec["processed_smiles"] for rec in records]
    logger.info("开始候选生成：%d 个分子；不使用 batch。", len(processed_smiles_list))

    per_mol_results = _parallel_or_sequential(
        processed_smiles_list,
        generate_candidates_for_smiles,
        config,
        desc="候选生成",
    )

    candidate_sources: Dict[str, Set[str]] = defaultdict(set)
    reached_global_limit = False
    global_duplicate_candidate_count = 0

    for result in per_mol_results:
        for smiles, sources in result.get("candidate_sources", {}).items():
            if config.max_candidates is not None and smiles not in candidate_sources and len(candidate_sources) >= config.max_candidates:
                reached_global_limit = True
                continue

            if smiles in candidate_sources:
                global_duplicate_candidate_count += 1
            candidate_sources[smiles].update(sources)

    if reached_global_limit:
        logger.warning("全局合并时达到 max_candidates=%s；后续新候选被跳过。", config.max_candidates)

    failed_count = sum(1 for r in per_mol_results if r.get("failed"))
    generation_stats = {
        "candidate_count_before_support_filter": len(candidate_sources),
        "reached_global_candidate_limit": reached_global_limit,
        "global_duplicate_candidate_count": int(global_duplicate_candidate_count),
        "failed_candidate_generation_molecule_count": int(failed_count),
        "avg_local_candidate_count": (
            sum(r["local_candidate_count"] for r in per_mol_results) / len(per_mol_results)
            if per_mol_results else 0
        ),
        "avg_explored_atom_set_count": (
            sum(r["explored_atom_set_count"] for r in per_mol_results) / len(per_mol_results)
            if per_mol_results else 0
        ),
        "avg_ring_completed_candidate_count": (
            sum(r["ring_completed_candidate_count"] for r in per_mol_results) / len(per_mol_results)
            if per_mol_results else 0
        ),
        "avg_skipped_existing_candidate_count_per_mol": (
            sum(r["skipped_existing_candidate_count"] for r in per_mol_results) / len(per_mol_results)
            if per_mol_results else 0
        ),
    }
    logger.info("候选生成完成：全局唯一候选数 %d。", len(candidate_sources))
    cleanup_memory()
    return candidate_sources, generation_stats

In [15]:
# =========================
# 6. 全局频率统计：support_mol_count 与 occurrence_count 分开，并按元素组成动态保留
# =========================
def safe_substruct_matches(mol: Chem.Mol, query_mol: Chem.Mol) -> Tuple[Tuple[int, ...], ...]:
    try:
        return mol.GetSubstructMatches(query_mol, uniquify=True, useChirality=False)
    except Exception:
        return tuple()


def count_one_fragment_frequency(args) -> Optional[Dict]:
    """统计单个候选片段的全局 support/occurrence；用于串行或 joblib 并行。"""
    smiles, sources, processed_smiles_list, n_mols, config = args
    query_mol = fragment_mol_from_smiles(smiles)
    if query_mol is None:
        return None

    support_mol_count = 0
    occurrence_count = 0

    # 在 worker 内从 processed_smiles 重建 mol，避免跨进程传递 RDKit Mol 的兼容性问题。
    for processed_smiles in processed_smiles_list:
        mol = Chem.MolFromSmiles(processed_smiles)
        if mol is None:
            continue
        matches = safe_substruct_matches(mol, query_mol)
        if matches:
            support_mol_count += 1
            occurrence_count += len(matches)

    support_ratio = support_mol_count / n_mols if n_mols else 0

    has_ring, has_aromatic_ring, has_aliphatic_ring = fragment_ring_flags(query_mol, smiles)
    elements = fragment_elements(query_mol)

    return {
        "smiles": smiles,
        "support_mol_count": support_mol_count,
        "occurrence_count": occurrence_count,
        "support_ratio": support_ratio,
        "elements": ";".join(elements),
        "num_atoms": query_mol.GetNumAtoms(),
        "num_bonds": query_mol.GetNumBonds(),
        # 兼容旧字段
        "is_ring": has_ring,
        "is_aromatic": has_aromatic_ring,
        # 新增更明确字段
        "has_aromatic_ring": has_aromatic_ring,
        "has_aliphatic_ring": has_aliphatic_ring,
        "source_types": ";".join(sorted(sources)),
    }


def count_fragment_frequencies(
    records: List[Dict],
    candidate_sources: Dict[str, Set[str]],
    config: FragmentConfig,
) -> pd.DataFrame:
    """对所有候选片段做全局匹配统计；筛选在缓存读取后单独执行。"""
    n_mols = len(records)
    processed_smiles_list = [rec["processed_smiles"] for rec in records]

    # 固定排序，保证同一候选集合下统计顺序可复现。
    candidate_smiles = sorted(candidate_sources.keys(), key=lambda s: (len(s), s))
    logger.info("开始全局频率统计：%d 个候选 × %d 个分子", len(candidate_smiles), n_mols)
    tasks = [
        (smiles, sorted(candidate_sources[smiles]), processed_smiles_list, n_mols, config)
        for smiles in candidate_smiles
    ]

    if JOBLIB_AVAILABLE and config.n_jobs != 1:
        logger.info("频率统计：使用 joblib 并行，n_jobs=%s，prefer=%s", config.n_jobs, config.parallel_prefer)
        rows = Parallel(n_jobs=config.n_jobs, prefer=config.parallel_prefer)(
            delayed(count_one_fragment_frequency)(task) for task in tasks
        )
    else:
        logger.info("频率统计：使用串行模式")
        rows = [count_one_fragment_frequency(task) for task in tasks]

    rows = [row for row in rows if row is not None]
    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df = df.sort_values(
        by=["support_mol_count", "occurrence_count", "num_atoms", "smiles"],
        ascending=[False, False, True, True],
    ).reset_index(drop=True)
    df.insert(0, "vocab_id", range(1, len(df) + 1))
    return df

In [16]:
# =========================
# 7. 结果保存与可选 Top-N 可视化
# =========================
def _jsonable_value(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple)):
        return [_jsonable_value(item) for item in value]
    if isinstance(value, dict):
        return {key: _jsonable_value(item) for key, item in value.items()}
    return value

def config_to_jsonable(config: FragmentConfig) -> Dict:
    data = asdict(config)
    return {key: _jsonable_value(value) for key, value in data.items()}



def _numeric_sort_column(df: pd.DataFrame, column: str) -> pd.Series:
    if column not in df.columns:
        return pd.Series([0] * len(df), index=df.index)
    return pd.to_numeric(df[column], errors="coerce").fillna(0)


def filter_cached_fragments_by_support(cached_df: pd.DataFrame, config: FragmentConfig, n_mols: int) -> Tuple[pd.DataFrame, Dict]:
    """按当前支持度比例重新过滤缓存结果，防止旧缓存绕过新阈值。"""
    work = cached_df.copy()
    support_counts = pd.to_numeric(work.get("support_mol_count"), errors="coerce").fillna(0).astype(int)

    threshold_types = []
    required_ratios = []
    required_counts = []
    elements_values = []
    has_ring_values = []
    has_aromatic_values = []
    has_aliphatic_values = []
    valid_smiles = []

    for smiles in work["smiles"].astype(str):
        frag_mol = fragment_mol_from_smiles(smiles)
        valid = frag_mol is not None
        valid_smiles.append(valid)
        if valid:
            threshold_type, required_ratio, required_count = support_threshold_for_fragment(frag_mol, n_mols, config)
            elements = ";".join(fragment_elements(frag_mol))
            has_ring, has_aromatic_ring, has_aliphatic_ring = fragment_ring_flags(frag_mol, smiles)
        else:
            threshold_type = "invalid_smiles"
            required_ratio = 1.0
            required_count = n_mols + 1
            elements = ""
            has_ring = False
            has_aromatic_ring = False
            has_aliphatic_ring = False
        threshold_types.append(threshold_type)
        required_ratios.append(required_ratio)
        required_counts.append(required_count)
        elements_values.append(elements)
        has_ring_values.append(has_ring)
        has_aromatic_values.append(has_aromatic_ring)
        has_aliphatic_values.append(has_aliphatic_ring)

    work["support_mol_count"] = support_counts
    work["support_ratio"] = support_counts / max(n_mols, 1)
    work["support_threshold_type"] = threshold_types
    work["required_support_ratio"] = required_ratios
    work["required_support_mol_count"] = required_counts
    work["elements"] = elements_values
    work["has_ring"] = has_ring_values
    work["has_aromatic_ring"] = has_aromatic_values
    work["has_aliphatic_ring"] = has_aliphatic_values

    keep_mask = pd.Series(valid_smiles, index=work.index) & (support_counts >= pd.Series(required_counts, index=work.index))
    filtered = work[keep_mask].copy()
    stats = {
        "input_molecule_count": int(n_mols),
        "cache_rows_before_support_filter": int(len(work)),
        "cache_rows_after_support_filter": int(len(filtered)),
        "cache_rows_removed_by_support_filter": int(len(work) - len(filtered)),
        "cache_support_filter_policy": {
            "CHON_only_min_ratio": config.chon_min_support_ratio,
            "contains_other_elements_min_ratio": config.other_min_support_ratio,
            "min_support_floor": config.min_support_floor,
        },
    }
    return filtered, stats


def frequency_cache_metadata(config: FragmentConfig) -> Dict:
    """返回决定候选集合与频率统计是否可复用的元数据（不含筛选阈值）。"""
    input_path = resolve_motif_path(config.input_file, config, must_exist=True)
    candidate_keys = [
        "min_size", "max_size", "max_candidates", "max_candidates_per_mol",
        "max_atom_sets_per_mol", "include_rings", "include_aromatic_rings",
        "complete_partial_rings", "complete_aromatic_rings",
        "complete_aliphatic_rings", "replace_ports_with_h",
    ]
    return {
        "generator_version": GENERATOR_VERSION,
        "input_file": str(input_path),
        "input_file_hash": hashlib.sha1(input_path.read_bytes()).hexdigest(),
        "candidate_config": {key: _jsonable_value(getattr(config, key)) for key in candidate_keys},
    }


def load_frequency_cache(config: FragmentConfig, n_mols: int) -> Tuple[Optional[pd.DataFrame], Dict]:
    """读取全量频率统计缓存，并按当前支持度阈值重新筛选。"""
    if not config.use_frequency_cache:
        return None, {"used_frequency_cache": False, "reason": "CONFIG.use_frequency_cache=False"}
    cache_path = resolve_motif_path(config.frequency_cache_file, config, must_exist=True)
    metadata_path = resolve_motif_path(config.frequency_cache_stats_file, config, must_exist=True)
    if not cache_path.exists() or not metadata_path.exists():
        return None, {"used_frequency_cache": False, "reason": "frequency cache or metadata file does not exist"}
    try:
        cached_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        cached_df = pd.read_csv(cache_path)
    except Exception as exc:
        return None, {"used_frequency_cache": False, "reason": repr(exc)}
    if cached_metadata != frequency_cache_metadata(config):
        return None, {"used_frequency_cache": False, "reason": "frequency cache metadata mismatch"}
    if "smiles" not in cached_df.columns or "support_mol_count" not in cached_df.columns:
        return None, {"used_frequency_cache": False, "reason": "frequency cache missing required columns"}
    filtered_df, filter_stats = filter_cached_fragments_by_support(cached_df, config, n_mols)
    filtered_df = filtered_df.sort_values(
        by=["support_mol_count", "occurrence_count", "num_atoms", "smiles"],
        ascending=[False, False, True, True],
    ).reset_index(drop=True)
    if "vocab_id" in filtered_df.columns:
        filtered_df = filtered_df.drop(columns=["vocab_id"])
    filtered_df.insert(0, "vocab_id", range(1, len(filtered_df) + 1))
    return filtered_df, {"used_frequency_cache": True, "frequency_cache_file": str(cache_path), **filter_stats}


def save_frequency_cache(full_frequency_df: pd.DataFrame, config: FragmentConfig) -> None:
    """保存未按支持度筛选的全量频率表及其可复用性元数据。"""
    cache_path = resolve_motif_path(config.frequency_cache_file, config)
    metadata_path = resolve_motif_path(config.frequency_cache_stats_file, config)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    full_frequency_df.to_csv(cache_path, index=False)
    metadata_path.write_text(json.dumps(frequency_cache_metadata(config), ensure_ascii=False, indent=2), encoding="utf-8")
    logger.info("全量频率统计缓存已保存: %s（%d 个候选）", cache_path, len(full_frequency_df))


def load_cached_fragment_results(config: FragmentConfig, n_mols: int) -> Tuple[Optional[pd.DataFrame], Dict]:
    """读取 results 中已有片段结果表作为缓存，避免重复计算。"""
    if not getattr(config, "use_result_cache", False):
        return None, {"used_result_cache": False, "reason": "CONFIG.use_result_cache=False"}

    # A canonical result is reusable only with matching generator/config metadata.
    stats_path = resolve_motif_path(config.stats_file, config, must_exist=True)
    try:
        cached_stats = json.loads(stats_path.read_text(encoding='utf-8'))
    except Exception:
        cached_stats = {}
    expected = {
        'generator_version': GENERATOR_VERSION,
        'max_candidates_per_mol': config.max_candidates_per_mol,
        'max_atom_sets_per_mol': config.max_atom_sets_per_mol,
        'chon_min_support_ratio': config.chon_min_support_ratio,
        'other_min_support_ratio': config.other_min_support_ratio,
    }
    if any(cached_stats.get(key) != value for key, value in expected.items()):
        return None, {'used_result_cache': False, 'reason': 'canonical cache metadata/version mismatch'}

    cache_paths = []
    for path_like in getattr(config, "result_cache_files", tuple()):
        path = resolve_motif_path(path_like, config, must_exist=True)
        if path.exists():
            cache_paths.append(path)

    if not cache_paths:
        return None, {"used_result_cache": False, "reason": "no configured cache CSV exists"}

    frames = []
    skipped = []
    for path in cache_paths:
        try:
            frame = pd.read_csv(path)
        except Exception as exc:
            skipped.append({"file": str(path), "reason": repr(exc)})
            continue
        if "smiles" not in frame.columns:
            skipped.append({"file": str(path), "reason": "missing smiles column"})
            continue
        frame = frame.copy()
        frame["cache_source_file"] = path.name
        frames.append(frame)

    if not frames:
        return None, {
            "used_result_cache": False,
            "reason": "cache files were unreadable or missing smiles column",
            "skipped_cache_files": skipped,
        }

    cached_df = pd.concat(frames, ignore_index=True, sort=False)
    cached_df["smiles"] = cached_df["smiles"].astype(str).str.strip()
    cached_df = cached_df[cached_df["smiles"] != ""].copy()

    cached_df, support_filter_stats = filter_cached_fragments_by_support(cached_df, config, n_mols)
    if cached_df.empty:
        return None, {
            "used_result_cache": False,
            "reason": "cache rows did not pass current support thresholds",
            "cache_files": [str(path) for path in cache_paths],
            "skipped_cache_files": skipped,
            **support_filter_stats,
        }

    before_dedup = len(cached_df)
    cached_df["_support_sort"] = _numeric_sort_column(cached_df, "support_mol_count")
    cached_df["_occurrence_sort"] = _numeric_sort_column(cached_df, "occurrence_count")
    cached_df = cached_df.sort_values(
        by=["_support_sort", "_occurrence_sort", "smiles"],
        ascending=[False, False, True],
    )
    cached_df = cached_df.drop_duplicates(subset=["smiles"], keep="first")
    cached_df = cached_df.drop(columns=["_support_sort", "_occurrence_sort"])

    for id_col in ["fragment_id", "vocab_id"]:
        if id_col in cached_df.columns:
            cached_df = cached_df.drop(columns=[id_col])
    cached_df = cached_df.reset_index(drop=True)
    cached_df.insert(0, "vocab_id", range(1, len(cached_df) + 1))

    stats = {
        "used_result_cache": True,
        "cache_files": [str(path) for path in cache_paths],
        "skipped_cache_files": skipped,
        **support_filter_stats,
        "cache_rows_before_dedup": int(before_dedup),
        "cache_rows_after_dedup": int(len(cached_df)),
        "dedup_key": "smiles",
        "dedup_policy": "keep row with highest support_mol_count, then highest occurrence_count",
    }
    return cached_df, stats

def save_results(df: pd.DataFrame, stats: Dict, config: FragmentConfig) -> None:
    output_file = resolve_motif_path(config.output_file, config)
    stats_file = resolve_motif_path(config.stats_file, config)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    stats_file.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(output_file, index=False)
    with open(stats_file, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    logger.info("vocab 结果已保存: %s", output_file)
    logger.info("运行统计已保存: %s", stats_file)


def save_top_fragment_grid(df: pd.DataFrame, config: FragmentConfig) -> Optional[Path]:
    """只保存 Top-N 片段图，默认不生成全量分页图。"""
    if df.empty:
        logger.warning("结果为空，跳过可视化。")
        return None

    viz_dir = resolve_motif_path(config.visualization_dir, config)
    viz_dir.mkdir(parents=True, exist_ok=True)
    top_df = df.head(config.top_n_viz)

    mols = []
    labels = []
    for _, row in top_df.iterrows():
        mol = fragment_mol_from_smiles(row["smiles"])
        if mol is None:
            continue
        mols.append(mol)
        labels.append(
            f"{row['vocab_id']}. {row['smiles']}\n"
            f"support={row['support_mol_count']}, occ={row['occurrence_count']}"
        )

    if not mols:
        logger.warning("没有可绘制的片段。")
        return None

    output_path = viz_dir / f"top_{len(mols)}_fragments.png"
    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=5,
        subImgSize=(300, 300),
        legends=labels,
        useSVG=False,
    )

    if hasattr(img, "save"):
        img.save(str(output_path))
    else:
        # 兼容某些环境下返回 image data 的情况
        data = getattr(img, "data", None)
        if isinstance(data, bytes):
            output_path.write_bytes(data)
        elif isinstance(data, str):
            output_path.write_text(data, encoding="utf-8")
        else:
            raise TypeError("Draw.MolsToGridImage 返回对象无法保存。")

    logger.info("Top-N 片段图已保存: %s", output_path)
    return output_path

In [17]:
# =========================
# 8. 主流程
# =========================
def run_pipeline(config: FragmentConfig = CONFIG) -> pd.DataFrame:
    setup_logging(config.log_level)
    logger.info("开始局域基元生成：并行阈值优化版。")
    logger.info("当前配置: %s", config_to_jsonable(config))
    logger.info("解析后的路径: %s", get_resolved_paths(config))

    records = load_molecule_records(config)
    if not records:
        raise ValueError("没有可用分子，请检查输入文件和 SMILES 列。")

    cached_df, cache_stats = load_frequency_cache(config, len(records))
    if cached_df is not None:
        logger.info("使用全量频率统计缓存，仅按当前阈值重新筛选。")
        final_stats = {
            "config": config_to_jsonable(config),
            "resolved_paths": get_resolved_paths(config),
            **cache_stats,
            "vocab_count_after_support_filter": int(len(cached_df)),
        }
        save_results(cached_df, final_stats, config)
        logger.info("最终保留 vocab 数: %d", len(cached_df))
        print(f"最终保留 vocab 数: {len(cached_df)}")
        display_cols = [col for col in [
            "vocab_id", "smiles", "support_mol_count", "support_ratio",
            "support_threshold_type", "required_support_mol_count",
            "occurrence_count", "elements", "num_atoms",
            "is_ring", "has_aromatic_ring", "has_aliphatic_ring", "cache_source_file",
        ] if col in cached_df.columns]
        if display_cols:
            display(cached_df[display_cols].head(10))
        return cached_df

    logger.info("未使用全量频率缓存，开始从输入 SMILES 重新计算。缓存状态: %s", cache_stats)
    candidate_sources, generation_stats = generate_candidate_fragments(records, config)
    full_frequency_df = count_fragment_frequencies(records, candidate_sources, config)
    if config.use_frequency_cache:
        save_frequency_cache(full_frequency_df, config)
    fragments_df, filter_stats = filter_cached_fragments_by_support(full_frequency_df, config, len(records))
    fragments_df = fragments_df.sort_values(
        by=["support_mol_count", "occurrence_count", "num_atoms", "smiles"],
        ascending=[False, False, True, True],
    ).reset_index(drop=True)
    if "vocab_id" in fragments_df.columns:
        fragments_df = fragments_df.drop(columns=["vocab_id"])
    fragments_df.insert(0, "vocab_id", range(1, len(fragments_df) + 1))

    final_stats = {
        "config": config_to_jsonable(config),
        "resolved_paths": get_resolved_paths(config),
        "input_molecule_count": len(records),
        "frequency_cache": cache_stats,
        "frequency_cache_written": str(resolve_motif_path(config.frequency_cache_file, config)),
        "full_frequency_candidate_count": int(len(full_frequency_df)),
        "support_filter": filter_stats,
        **generation_stats,
        "vocab_count_after_support_filter": int(len(fragments_df)),
        "output_vocab_count": int(len(fragments_df)),
        "generator_version": GENERATOR_VERSION,
        "max_candidates_per_mol": config.max_candidates_per_mol,
        "max_atom_sets_per_mol": config.max_atom_sets_per_mol,
        "chon_min_support_ratio": config.chon_min_support_ratio,
        "other_min_support_ratio": config.other_min_support_ratio,
        "input_file": str(resolve_motif_path(config.input_file, config)),
        "input_file_hash": hashlib.sha1(resolve_motif_path(config.input_file, config).read_bytes()).hexdigest(),
        "support_filter_policy": {
            "CHON_only": {
                "definition": "片段显式原子元素集合只包含 C/H/O/N。RDKit 隐式 H 不作为独立原子统计。",
                "required_support_ratio": config.chon_min_support_ratio,
                "required_support_mol_count": max(config.min_support_floor, math.ceil(len(records) * config.chon_min_support_ratio)),
            },
            "contains_other_elements": {
                "definition": "片段显式原子中包含 C/H/O/N 之外的元素。",
                "required_support_ratio": config.other_min_support_ratio,
                "required_support_mol_count": max(config.min_support_floor, math.ceil(len(records) * config.other_min_support_ratio)),
            },
        },
        "frequency_definition": {
            "support_mol_count": "包含该片段的分子数。",
            "occurrence_count": "该片段在所有分子中的总匹配次数，使用 RDKit GetSubstructMatches(..., uniquify=True)。",
        },
        "parallel": {
            "joblib_available": JOBLIB_AVAILABLE,
            "n_jobs": config.n_jobs,
            "prefer": config.parallel_prefer,
        },
    }

    save_results(fragments_df, final_stats, config)
    logger.info("最终保留 vocab 数: %d", len(fragments_df))
    print(f"最终保留 vocab 数: {len(fragments_df)}")

    if config.visualize:
        save_top_fragment_grid(fragments_df, config)
    else:
        logger.info("visualize=False，已跳过片段可视化。")

    if not fragments_df.empty:
        logger.info("Top 10 片段预览：")
        display_cols = [
            "vocab_id", "smiles", "support_mol_count", "support_ratio",
            "support_threshold_type", "required_support_mol_count",
            "occurrence_count", "elements", "num_atoms",
            "is_ring", "has_aromatic_ring", "has_aliphatic_ring",
        ]
        display(fragments_df[display_cols].head(10))
    else:
        logger.warning("最终结果为空。可以尝试检查输入 SMILES、增大候选生成上限或降低比例阈值。")

    return fragments_df

In [ ]:
# =========================
# 9. 运行
# =========================
# 直接运行本 cell 即可。
# 默认假定本 notebook/脚本位于 LCMWR/scripts 下：
#   输入：../data/unique_smiles_for_fragments.csv
#   输出：../results/local_fragments_parallel_threshold.csv
#
# 如需修改参数，例如：
# CONFIG.n_jobs = -1
# CONFIG.chon_min_support_ratio = 0.01
# CONFIG.other_min_support_ratio = 0.005
# CONFIG.visualize = True
# 如果路径仍无法自动识别，可手动指定：
# CONFIG.motif_root = Path(r"E:/your_project/motif")

vocab_df = run_pipeline(CONFIG)
